# Study 09 · 08 · E7 · A matching statistical model (NB2 regression)

Does the null result hold outside boosted trees? NB2 regression with the persistence offset and the same case-history and geography terms (`run_statistical_benchmarks.fit_nb`: log1p of the case terms, standardisation, ridge fallback if needed), with and without **exactly** the b3 climate block (lag 2, linear terms). This replaces the paper's NB2 climate arm, which used linear and quadratic terms in four temperature windows. The NB2 GLM is deterministic: one fit per block.

In [1]:
import os
import sys
from pathlib import Path

os.environ.update({"DENGUEKAY_XGB_DEVICE": "cpu", "DENGUEKAY_XGB_NJOBS": "2", "OMP_NUM_THREADS": "2",
                   "OPENBLAS_NUM_THREADS": "1", "VECLIB_MAXIMUM_THREADS": "1", "MLFLOW_DISABLE_AGENT_HINT": "1"})
STUDY = Path.cwd().parent
sys.path.insert(0, str(STUDY / "code"))

import numpy as np
import pandas as pd

import common as c

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)
import matplotlib.pyplot as plt

import experiments as ex
import run_extra as rx

FIG = STUDY / "figures"
RES = c.OUTPUT / "resultados"
FIG.mkdir(exist_ok=True)
RES.mkdir(exist_ok=True)
base = pd.read_csv(c.OUTPUT / "baselines.csv.gz", dtype={"ubigeo": str}, parse_dates=["semana_inicio", "origen_cierre"])
cols = ["semilla", "bloque", "horizonte", "ubigeo", "semana_inicio", "casos_Dengue", "brote", "probabilidad_calibrada", "conteo_publicado", "persistencia"]
nb = {n: c.read_arm(n, seeds=(17,), columns=cols, folder=rx.NB2_DIR) for n in ("nb_b3", "nb_historia_geo")}
fits = {n: pd.read_csv(rx.NB2_DIR / f"{n}_s17" / "ajustes.csv") for n in nb}
for n, f in fits.items():
    flags = f.filter(like="convergio")
    print(n, "| blocks x horizons converged without the ridge fallback:", int(flags.to_numpy().sum()), "of", flags.size)

nb_b3 | blocks x horizons converged without the ridge fallback: 52 of 60
nb_historia_geo | blocks x horizons converged without the ridge fallback: 57 of 60


In [2]:
a, b = ex.sse(nb["nb_b3"]), ex.sse(nb["nb_historia_geo"])
pp = ex.sse(base, "persistencia_prob")
xgb = ex.sse(c.read_arm("historia_geo", columns=ex.PRED_COLUMNS))
e7 = pd.concat([ex.contrast(a, b).assign(comparison="NB2 + b3 climate vs NB2 history+geography"),
                ex.contrast(b, pp).assign(comparison="NB2 history+geography vs probabilistic persistence"),
                ex.contrast(b, xgb).assign(comparison="NB2 history+geography vs XGBoost history+geography")])
e7 = e7[["comparison", "horizonte", "brier_a", "brier_b", "cambio_agrupado_%", "ic95_agrupado", "cambio_igual_peso_%", "temporadas_a_mejor"]]
e7.to_csv(RES / "e7_nb2.csv", index=False)
e7.round(3)

,comparison,horizonte,brier_a,brier_b,cambio_agrupado_%,ic95_agrupado,cambio_igual_peso_%,temporadas_a_mejor
0,NB2 + b3 climate vs NB2 history+geography,2,0.030,0.029,2.955,"(-2.25, 11.05)",-1.319,11
1,NB2 + b3 climate vs NB2 history+geography,3,0.035,0.034,3.511,"(-3.11, 12.83)",-2.256,11
2,NB2 + b3 climate vs NB2 history+geography,4,0.039,0.038,3.645,"(-3.98, 13.85)",-2.509,11
0,NB2 history+geography vs probabilistic persistence,2,0.029,0.029,0.572,"(-4.42, 8.09)",10.056,5
1,NB2 history+geography vs probabilistic persistence,3,0.034,0.033,0.964,"(-5.66, 10.76)",13.660,6
2,NB2 history+geography vs probabilistic persistence,4,0.038,0.037,1.287,"(-5.98, 12.46)",19.318,6
0,NB2 history+geography vs XGBoost history+geography,2,0.029,0.028,1.538,"(-0.69, 4.53)",0.297,8
1,NB2 history+geography vs XGBoost history+geography,3,0.034,0.032,4.034,"(1.08, 7.48)",3.791,6
2,NB2 history+geography vs XGBoost history+geography,4,0.038,0.036,6.397,"(3.52, 9.71)",7.320,1
